<a href="https://colab.research.google.com/github/eltioxkypper/curso_ai_llama/blob/main/Hands_On_Fundamentos_de_LLMs_con_Modelos_Llama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: FUNDAMENTOS DE LLMS CON MODELOS LLAMA**


Una vez vista la masterclass ***Fundamentos de LLMs y Arquitectura de Llama***, se proporciona el siguiente ***Colab*** para construir, en vivo, la primera llamada a Llama y medir su comportamiento.

Usamos **Groq** para tener acceso a inferencia de Llama con GPU gratuita, sin necesidad de infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1bjMHw7fmMxo9dyQp140e9mqotKRkfKBg?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Antes de llamar a Llama necesitamos activar la GPU gratuita de Colab y conectar nuestra cuenta de **Groq**. La librería `groq` es el cliente oficial en Python para hacer llamadas a la API.

### **COLAB SECRETS**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [ ]:
# Instalar cliente de Groq y leer API key desde Colab Secrets
!pip install groq -q

In [ ]:
import os
from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get("GROQ_API_KEY"))
print("GROQ client initializing correctly....")

GROQ client initializing correctly....


### **DE PALABRAS A TOKENS**

Antes de que Llama genere una sola palabra, tu prompt se divide en **tokens**. Vamos a definir un prompt de ejemplo que después enviaremos a Llama.

In [ ]:
# Definir un prompt de ejemplo (una pregunta real de tu propio contexto)
prompt = 'Cual es la diferencia entre la RAM y el almacenamiento en una computadora?'

print(prompt)

Cual es la diferencia entre la RAM y el almacenamiento en una computadora?


### **PRIMERA LLAMADA A LLAMA (ZERO-SHOT)**

Un LLM predice el siguiente token más probable, **no busca la respuesta en una base de datos**. Vamos a enviar el prompt a Llama en modo zero-shot (sin ejemplos previos) y ver la respuesta generada.

In [ ]:
# Enviar el prompt a Llama en Groq y mostrar la respuesta generada
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print(response.choices[0].message.content)

## RAM vs. Almacenamiento: ¿Qué las diferencia realmente?

| Característica | RAM (Memoria de Acceso Aleatorio) | Almacenamiento (HDD, SSD, NVMe, etc.) |
|----------------|-----------------------------------|----------------------------------------|
| **Tipo de memoria** | Volátil (se borra al apagar el equipo) | No volátil (mantiene datos sin energía) |
| **Uso principal** | Mantener los datos que el CPU necesita *en ese instante* | Guardar el sistema operativo, programas, archivos y datos a largo plazo |
| **Velocidad** | Muy alta (latencia de 10–100 ns, ancho de banda de varios GB/s) | Más lenta (latencia de microsegundos a milisegundos, ancho de banda de 0.1–5 GB/s) |
| **Capacidad típica** | 4 GB – 64 GB (o más en servidores) | 120 GB – varios TB (HDD) o 500 GB – 4 TB (SSD/NVMe) |
| **Costo por GB** | Muy alto (USD 5–15 / GB) | Muy bajo (USD 0.02–0.10 / GB) |
| **Ubicación** | En la placa base (chip RAM) | En la placa base (controlador) o en unidades externas |
| **Acceso** | Dire

In [ ]:
print(response.model_dump_json(indent=2))

### **CUÁNTOS TOKENS CONSUMIÓ EL PROMPT**

La respuesta de la API trae un resumen (`usage`) con el número de tokens de entrada y de salida. Es la forma más directa de responder: ¿cuántos tokens consumió mi prompt?

In [ ]:
# Mostrar cuántos tokens tuvo el prompt y cuántos tuvo la respuesta
print("Tokends del prompt:", response.usage.prompt_tokens)
print("Tokends de la respuesta:", response.usage.completion_tokens)
print("Total de tokens:", response.usage.total_tokens)

### **MIDIENDO LA LATENCIA DE INFERENCIA**

Elegir el tamaño correcto de modelo es una decisión de ingeniería, no solo de potencia bruta. Vamos a medir cuánto tarda Llama en responder el mismo prompt.

In [ ]:
# Medir el tiempo de respuesta de Llama para el mismo prompt
import time

init = time.time()
response_time = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

duration = time.time() - init

print(f"Response time: {duration:.2f} seconds")

### **COMPARANDO DOS TAMAÑOS DE MODELO**

Repite la misma llamada usando la versión más grande de Llama disponible en Groq y compara tiempo de respuesta y calidad contra el modelo ligero.

In [ ]:
# Repetir la llamada con un modelo Llama más grande y comparar tiempo y calidad
inicio = time.time()

response_grande = client.chat.completions.create(
model="llama-3.3-70b-versatile",
messages=[{"role": "user", "content": prompt}]
)

duration_long = time.time() - inicio

print(f"Modelo ligero: {duration:.2f} s — {response.usage.total_tokens} tokens")
print(f"Modelo grande: {duration_long:.2f} s — {response_grande.usage.total_tokens} tokens")
print("\nRespuesta del modelo grande:\n", response_grande.choices[0].message.content)

## **CHALLENGE: COMPARADOR DE MODELOS LLAMA**

Una vez visto el ***Hands-On: Fundamentos de LLMs***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño **comparador que envíe 3 preguntas** reales a Llama y registre, para cada una, el **tiempo de respuesta** y el **número de tokens** consumidos.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

### **INSTRUCCIONES:**

**1. Carga la API key y genera 3 preguntas:**

   * Lee la API key de llama ya configurada desde **Colab Secrets**.
   * Construye una lista vacía llamada `preguntas` y agrégale 3 preguntas frecuentes de tu propio contexto (soporte técnico, tienda, escuela, etc.).

In [ ]:
# Leer API key desde Colab Secrets
!pip install groq -q
import os
from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get("GROQ_API_KEY"))
print("GROQ client initializing correctly....")

In [ ]:
# Definir la lista de preguntas
preguntas = [
    "Cuáles son las universidades mejores rankeadas de méxico este 2026",
    "En que consiste el problema SNAT en Microsoft Azure",
    "Cuál es la diferencia entre bases de datos relacionales y no relacionales"
]


def model_response(prompt):
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

**2. Consulta la primera pregunta:** Envía la primera pregunta a Llama usando el modelo más ligero disponible en Groq. Guarda la respuesta, el tiempo de respuesta y el número de tokens en un diccionario llamado `resultado_1`.

In [ ]:
modelos = client.models.list()
for m in modelos.data:
    print(m.id)

In [ ]:
# Consultar la primera pregunta y guardar el resultado en resultado_1
resultado_1 = model_response(preguntas[0])
print(resultado_1)


**3. Repite para la segunda y tercera pregunta:** Crea `resultado_2` y `resultado_3` de la misma forma.

In [ ]:
# Consultar la segunda pregunta y guardar el resultado en resultado_2
resultado_2 = model_response(preguntas[1])
print(resultado_2)

In [ ]:
# Consultar la tercera pregunta y guardar el resultado en resultado_3
resultado_3 = model_response(preguntas[2])
print(resultado_3)

**4. Junta los resultados:** Agrega `resultado_1`, `resultado_2` y `resultado_3` a una lista vacía llamada `resultados`.

In [ ]:
# Definir la lista resultados y agregar los tres diccionarios
resultados = []
resultados.append(resultado_1)
resultados.append(resultado_2)
resultados.append(resultado_3)

**5. Muestra la comparación:** Imprime `resultados` y concluye si el modelo ligero resolvió las 3 preguntas satisfactoriamente.

In [ ]:
# Mostrar la tabla final de resultados
for resultado in resultados:
  print("----------------------------------------------")
  print(resultado)

El modelo utilizado me permitio obtener los dstos correspondientes, por lo que concluyo que si funciona.